# Tarea 2: Segmentación de Clientes
## Notebook 04 — Customer Segmentation using KMeans + PCA

**Objetivo:** Segmentar la base de clientes en grupos homogéneos para orientar la estrategia comercial de easyMoney.

**Metodología:**
- Proceso iterativo de feature engineering en 3 pasos documentados
- KMeans++ con selección de k óptimo (Elbow + Silhouette + Davies-Bouldin + Calinski-Harabasz)
- PCA para visualización 2D orientativa
- Profiling de clusters para interpretación de negocio

**Input:** `master_df_flags.parquet`
**Output:** `customer_segments.csv`, `cluster_profiles.csv`

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import os
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
K_FINAL = 7  # k optimo — justificado en Seccion 7.3

print("All imports OK")

## 2. Carga de Datos

In [ ]:
df = pd.read_parquet('../../data/processed/master_df_flags.parquet')
df_latest = df[df['pk_partition'] == df['pk_partition'].max()]

print(f"Shape (todas las particiones): {df.shape}")
print(f"Shape (ultimo snapshot):       {df_latest.shape}")
print(f"Clientes unicos:               {df_latest['pk_cid'].nunique()}")
print(f"Particion seleccionada:        {df_latest['pk_partition'].unique()}")
df_latest.head(3)

### Nota sobre el universo de clientes

El dataset historico contiene **456,318 clientes unicos** a lo largo de todas las particiones.
El ultimo snapshot (mayo 2019) tiene **442,995 clientes** — una diferencia de **13,323 clientes**
que no aparecen en la ultima particion, indicando probable churn antes del corte de datos.

La segmentacion usa exclusivamente el ultimo snapshot porque:
- Representa el estado actual de cada cliente
- Evita duplicados (un cliente por fila)
- Es el universo accionable para campanas de marketing

## 3. Exploración Inicial del Dataset

In [ ]:
print("Columnas disponibles:")
print(df_latest.columns.tolist())

print()
nulos = df_latest.isnull().sum()
print("Nulos por columna:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin nulos en el dataset")

print()
print("Valores unicos de variables categoricas clave:")
for col in ['gender', 'segment', 'age_group', 'salary_group']:
    vals = sorted(df_latest[col].dropna().unique())
    print(f"  {col}: {vals}")

## 4. Selección de Features Base

Se definen dos grupos de features:

**Productos (binarias 0/1):** portfolio de productos de cada cliente.
**Comportamentales y demograficas:** antiguedad, actividad, edad, salario, genero.

> **Nota:** `em_acount` (sin segunda 'c') es el nombre original en la fuente de datos.
> Se mantiene para coherencia con el pipeline upstream — no es un error del notebook.

In [ ]:
product_cols = [
    'em_acount', 'em_account_p', 'emc_account',
    'payroll_account', 'payroll',
    'credit_card', 'debit_card',
    'funds', 'securities', 'pension_plan',
    'long_term_deposit', 'short_term_deposit',
    'loans', 'mortgage'
]

feature_cols = product_cols + ['age', 'salary', 'active_customer',
                                'total_products', 'client_age_months']

# Validacion de valores de gender antes del encoding
unique_genders = df_latest['gender'].dropna().unique()
print(f"Valores unicos de gender: {sorted(unique_genders)}")

gender_map = {'H': 0, 'V': 1}  # H = Hombre, V = Varon
unexpected = set(df_latest['gender'].dropna().unique()) - set(gender_map.keys())
if unexpected:
    print(f"AVISO: valores inesperados en gender: {unexpected} -> mapeados a 0 por defecto")
else:
    print("gender OK: H (Hombre=0), V (Varon=1) — sin valores inesperados")

# DataFrame de clustering
df_clust = df_latest[['pk_cid'] + feature_cols].copy()
df_clust['gender_enc'] = df_latest['gender'].map(gender_map).fillna(0).astype(int)
df_clust['active_customer'] = df_clust['active_customer'].astype(int)

feature_cols_enc = feature_cols + ['gender_enc']
scaler = StandardScaler()

print()
print(f"Features base para clustering: {len(feature_cols_enc)}")
print(f"Shape: {df_clust[feature_cols_enc].shape}")
df_clust[feature_cols_enc].describe().round(2)

## 5. Proceso Iterativo de Feature Engineering

Se aplico un proceso de **3 iteraciones** hasta obtener una distribucion de clusters estable y accionable:

| Iteracion | Tratamiento aplicado | Problema detectado |
|-----------|---------------------|--------------------|
| 1 | Features originales sin tratamiento | Clusters degenerados por outliers extremos en salary y age |
| 2 | Capping al percentil 99 de salary y age | Mini-clusters persistentes por features con varianza casi nula |
| 3 | Eliminacion de features de baja varianza | Distribucion estable — configuracion adoptada |

### 5.1 Iteración 1 — Escalado Estándar sin Tratamiento de Outliers

Primera ejecucion sobre las 20 features originales escaladas con StandardScaler.
El objetivo es diagnosticar el comportamiento inicial del modelo.

In [ ]:
X1_scaled = scaler.fit_transform(df_clust[feature_cols_enc].values)

km1 = KMeans(n_clusters=K_FINAL, init='k-means++', random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = km1.fit_predict(X1_scaled)

dist1 = df_clust['cluster'].value_counts().sort_index()
print("Distribucion — Iteracion 1 (sin tratamiento de outliers):")
for c, n in dist1.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia: {km1.inertia_:,.0f}")

small_1 = [(c, int(n)) for c, n in dist1.items() if n < 100]
if small_1:
    print()
    print(f"AVISO: clusters con < 100 clientes detectados: {small_1}")
    print("  -> Probable distorsion por valores extremos en salary o age")

### 5.2 Análisis de Outliers — salary y age

Los clusters con muy pocos clientes se forman porque KMeans asigna centroides propios
a los valores extremos (salary de millones de euros, edades de 90-105 anos).
Se investigan los clusters problemáticos y la distribucion en percentiles altos.

In [ ]:
# Analisis de clusters con < 100 clientes
small_clusters_1 = [c for c, n in dist1.items() if n < 100]

for c in small_clusters_1:
    mask = df_clust['cluster'] == c
    n = int(mask.sum())
    print(f"=== Cluster {c} ({n} clientes) ===")
    print(df_clust[mask][['age', 'salary', 'total_products', 'client_age_months']].describe().round(2))
    print()

print("=== Distribucion salary (percentiles altos) ===")
q_salary = df_clust['salary'].quantile([0.95, 0.99, 0.999, 1.0])
for q, v in q_salary.items():
    print(f"  p{q*100:.1f}%: {v:>15,.0f} EUR")

print()
print("=== Distribucion age (percentiles altos) ===")
q_age = df_clust['age'].quantile([0.95, 0.99, 0.999, 1.0])
for q, v in q_age.items():
    print(f"  p{q*100:.1f}%: {v:.0f} anos")

print()
print("Conclusion: salary max ~28.9M EUR (p99=424k EUR) y age max 105 anos (p99=74 anos).")
print("Se aplicara capping al percentil 99 en ambas variables.")

### 5.3 Iteración 2 — Capping al Percentil 99

Se limitan salary y age a su percentil 99 para que los valores extremos
no acaparen centroides propios y distorsionen el clustering.

In [ ]:
p99_salary = df_clust['salary'].quantile(0.99)
p99_age    = df_clust['age'].quantile(0.99)

df_clust['salary_capped'] = df_clust['salary'].clip(upper=p99_salary)
df_clust['age_capped']    = df_clust['age'].clip(upper=p99_age)

feature_cols_capped = [c for c in feature_cols_enc
                       if c not in ['salary', 'age']] + ['salary_capped', 'age_capped']

print(f"Cap salary -> {p99_salary:,.0f} EUR  (maximo original: {df_clust['salary'].max():,.0f} EUR)")
print(f"Cap age    -> {p99_age:.0f} anos   (maximo original: {df_clust['age'].max():.0f} anos)")
print()
print(f"Features con capping ({len(feature_cols_capped)}): {feature_cols_capped}")

X2_scaled = scaler.fit_transform(df_clust[feature_cols_capped].values)

km2 = KMeans(n_clusters=K_FINAL, init='k-means++', random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = km2.fit_predict(X2_scaled)

dist2 = df_clust['cluster'].value_counts().sort_index()
print()
print("Distribucion — Iteracion 2 (capping al p99):")
for c, n in dist2.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia: {km2.inertia_:,.0f}")

small_2 = [(c, int(n)) for c, n in dist2.items() if n < 100]
if small_2:
    print()
    print(f"AVISO: siguen existiendo clusters pequenos: {small_2}")
    print("  -> Investigar features de baja varianza en la siguiente iteracion")

### 5.4 Análisis de Clusters Pequeños y Features con Baja Varianza

Con el capping aplicado siguen apareciendo mini-clusters. Se investigan:
1. El perfil detallado de los clusters pequeños restantes
2. La prevalencia de cada producto para identificar features con varianza casi nula

In [ ]:
# Perfil de clusters pequenos en Iteracion 2
small_clusters_2 = [c for c, n in dist2.items() if n < 100]

print("=== Perfil de clusters pequenos (Iteracion 2) ===")
for c in small_clusters_2:
    mask = df_clust['cluster'] == c
    n = int(mask.sum())
    print(f"--- Cluster {c} ({n} clientes) ---")
    if n <= 10:
        print(df_clust[mask][feature_cols_capped].T.to_string())
    else:
        print(df_clust[mask][feature_cols_capped].describe().round(2).to_string())
    print()

# Prevalencia de cada producto
print("=== Prevalencia de productos (% de clientes con producto = 1) ===")
low_var_candidates = []
for col in product_cols:
    pct = df_clust[col].mean() * 100
    flag = "  <- BAJA VARIANZA" if pct < 0.5 else ""
    print(f"  {col:<25}: {pct:>7.3f}%{flag}")
    if pct < 0.5:
        low_var_candidates.append(col)

print()
print(f"Features con prevalencia < 0.5%: {low_var_candidates}")
print("Estas features no aportan poder discriminante y generan mini-clusters.")
print("-> Se eliminaran en la Iteracion 3.")

### 5.5 Iteración 3 — Configuración Final (Eliminación de Features de Baja Varianza)

Se eliminan las 4 features con prevalencia < 0.5%.
Esta es la configuracion definitiva adoptada para todo el analisis posterior.

In [ ]:
low_variance_cols = ['em_account_p', 'short_term_deposit', 'loans', 'mortgage']

feature_cols_final = [c for c in feature_cols_capped if c not in low_variance_cols]

continuous_cols = ['age_capped', 'salary_capped', 'client_age_months']
binary_cols     = [c for c in feature_cols_final if c not in continuous_cols]

print(f"Features eliminadas ({len(low_variance_cols)}): {low_variance_cols}")
print()
print(f"Features finales ({len(feature_cols_final)}):")
print(f"  Continuas    ({len(continuous_cols)}): {continuous_cols}")
print(f"  Binarias/cnt ({len(binary_cols)}): {binary_cols}")

X_final_scaled = scaler.fit_transform(df_clust[feature_cols_final].values)
print()
print(f"Matriz X_final_scaled: {X_final_scaled.shape}")

km3 = KMeans(n_clusters=K_FINAL, init='k-means++', random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = km3.fit_predict(X_final_scaled)

dist3 = df_clust['cluster'].value_counts().sort_index()
print()
print("Distribucion — Iteracion 3 (configuracion final):")
for c, n in dist3.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia: {km3.inertia_:,.0f}")
min_cluster = int(dist3.min())
print(f"Cluster minimo: {min_cluster:,} clientes")
if min_cluster >= 1000:
    print("OK: distribucion estable — configuracion adoptada como definitiva")
else:
    print("AVISO: revisar cluster pequeno")

## 6. Resumen de la Configuración Final de Features

| Decision | Detalle | Motivo |
|----------|---------|--------|
| Capping salary | Percentil 99 → max 424,306 EUR | Salario max original ~28.9M EUR generaba centroides espurios |
| Capping age | Percentil 99 → max 74 anos | Edad max original 105 anos generaba centroides espurios |
| Features eliminadas | `em_account_p`, `short_term_deposit`, `loans`, `mortgage` | Prevalencia < 0.05% — sin poder discriminante |
| Escalado | StandardScaler sobre todas las features | Equipara escala; da peso relativo a productos con baja prevalencia |
| Features finales | **16 features** | 10 productos + 3 continuas + genero + actividad + total_products |

## 7. Selección del Número Óptimo de Clusters

Con la configuracion final (`X_final_scaled`, 16 features) se busca el k optimo
combinando cuatro criterios complementarios:

- **Elbow (inercia WCSS):** detecta el punto de inflexion donde anadir k da rendimientos decrecientes
- **Silhouette Score (mayor = mejor):** cohesion intra-cluster vs separacion inter-cluster
- **Davies-Bouldin (menor = mejor):** similitud media entre cada cluster y el mas parecido
- **Calinski-Harabasz (mayor = mejor):** ratio dispersion inter-cluster / intra-cluster

### 7.1 Elbow + Silhouette Score

In [ ]:
k_range = range(2, 11)
inertias    = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_final_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_final_scaled, labels, sample_size=50000, random_state=RANDOM_STATE)
    silhouettes.append(sil)
    print(f"k={k:2d}  |  inertia={km.inertia_:>12,.0f}  |  silhouette={sil:.4f}")

print()
print("Analisis completado")

### 7.2 Métricas Adicionales de Validación

Se evaluan Davies-Bouldin y Calinski-Harabasz sobre una muestra de 50,000 clientes.
El silhouette de la celda anterior (con igual random seed) se usa para comparacion directa.

In [ ]:
np.random.seed(RANDOM_STATE)
sample_idx_val = np.random.choice(len(X_final_scaled), size=50000, replace=False)
X_sample = X_final_scaled[sample_idx_val]

print("=== Metricas de Validacion — k=2..10 ===")
print()
results = []
for k, inertia, sil in zip(k_range, inertias, silhouettes):
    km = KMeans(n_clusters=k, init='k-means++', random_state=RANDOM_STATE, n_init=10)
    labels_sample = km.fit_predict(X_sample)
    db = davies_bouldin_score(X_sample, labels_sample)
    ch = calinski_harabasz_score(X_sample, labels_sample)
    results.append({'k': k, 'silhouette': sil, 'davies_bouldin': db, 'calinski_harabasz': ch})
    print(f"k={k:2d}  |  silhouette={sil:.4f}  |  davies_bouldin={db:.4f}  |  calinski_harabasz={ch:,.0f}")

df_validation = pd.DataFrame(results)

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=('Silhouette (mayor = mejor)',
                                    'Davies-Bouldin (menor = mejor)',
                                    'Calinski-Harabasz (mayor = mejor)'))

for col_name, color, col_pos in [('silhouette', 'steelblue', 1),
                                   ('davies_bouldin', 'darkorange', 2),
                                   ('calinski_harabasz', 'green', 3)]:
    fig.add_trace(
        go.Scatter(x=df_validation['k'], y=df_validation[col_name],
                   mode='lines+markers', marker=dict(size=8, color=color)),
        row=1, col=col_pos
    )

fig.add_vline(x=7, line_dash='dash', line_color='red',
              annotation_text='k=7', annotation_position='top right')

fig.update_layout(title='Validacion del numero optimo de clusters — 3 metricas',
                  height=420, showlegend=False)
fig.update_xaxes(title_text='k')
fig.show()
print()
print("Validacion completada")

### 7.3 Justificación de k=7

| Metrica | Mejor k matematico | Valor en k=7 | Valoracion |
|---------|-------------------|--------------|------------|
| Silhouette (mayor) | k=6: ~0.256 | ~0.221 | Aceptable — ligero trade-off por granularidad de negocio |
| Davies-Bouldin (menor) | k=10: ~1.056 | ~1.289 | Aceptable |
| Calinski-Harabasz (mayor) | k=2: ~13,203 | ~9,388 | Estable en el rango k=5..10 |
| Elbow | Sin codo claro | — | Habitual en datos bancarios sin clusters naturales muy separados |

**Decision final: k=7** justificada por cuatro razones:

1. **Requisito de negocio (Carol):** "7 u 8 grupos" — k=7 satisface directamente el mandato comercial
2. **Interpretabilidad:** 7 segmentos ofrecen suficiente granularidad para disear acciones diferenciadas sin fragmentacion excesiva
3. **Estabilidad estadistica:** con k=7 el cluster minimo supera los 1,000 clientes — cada segmento tiene masa critica accionable
4. **Ausencia de k dominante:** el silhouette local en k=6 (0.256) y k=9 (0.279) es marginalmente superior, pero sin mejora estructural que justifique desobedecer el requisito de negocio

## 8. Visualización Elbow + Silhouette

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Metodo del Codo (Elbow)',
                                    'Silhouette Score'))

fig.add_trace(
    go.Scatter(x=list(k_range), y=inertias,
               mode='lines+markers',
               marker=dict(size=8, color='steelblue'),
               line=dict(width=2), name='Inertia'),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=list(k_range), y=silhouettes,
               mode='lines+markers',
               marker=dict(size=8, color='darkorange'),
               line=dict(width=2), name='Silhouette'),
    row=1, col=2
)

fig.add_vline(x=7, line_dash='dash', line_color='red',
              annotation_text='k=7 seleccionado', annotation_position='top right')

fig.update_layout(title='Seleccion del numero optimo de clusters',
                  height=450, width=900, showlegend=False)
fig.update_xaxes(title_text='Numero de clusters (k)')
fig.update_yaxes(title_text='Inertia (WCSS)', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2)
fig.show()

## 9. Modelo Final — KMeans con k=7

Se entrena el modelo definitivo sobre `X_final_scaled` (16 features, capping al p99):
- `n_init=20`: 20 inicializaciones aleatorias, se conserva la de menor inercia
- `init='k-means++'`: inicializacion inteligente de centroides para convergencia estable
- `random_state=42`: reproducibilidad garantizada

In [ ]:
kmeans_final = KMeans(n_clusters=K_FINAL, init='k-means++',
                      random_state=RANDOM_STATE, n_init=20)
df_clust['cluster'] = kmeans_final.fit_predict(X_final_scaled)

dist_final = df_clust['cluster'].value_counts().sort_index()
print("Distribucion final de clientes por cluster:")
for c, n in dist_final.items():
    print(f"  Cluster {c}: {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

print()
print(f"Inertia final:        {kmeans_final.inertia_:,.0f}")
print(f"Cluster mas pequeno:  {int(dist_final.min()):,} clientes")
print(f"Cluster mas grande:   {int(dist_final.max()):,} clientes")
print()
print("Modelo final entrenado correctamente")

## 10. Profiling de Clusters

Se calcula la media de cada feature por cluster para interpretar el perfil de cada segmento.

In [ ]:
profile_cols = ['em_acount', 'emc_account', 'payroll_account', 'payroll',
                'credit_card', 'debit_card', 'funds', 'securities',
                'pension_plan', 'long_term_deposit', 'active_customer',
                'total_products', 'salary_capped', 'age_capped',
                'client_age_months']

profile = df_clust.groupby('cluster')[profile_cols].mean().round(3)
profile['n_clientes']   = df_clust['cluster'].value_counts().sort_index()
profile['pct_clientes'] = (profile['n_clientes'] / len(df_clust) * 100).round(1)

print("=== PERFIL COMPLETO POR CLUSTER ===")
print()
print(profile.T.to_string())

## 11. Asignación de Nombres a los Segmentos

In [ ]:
cluster_names = {
    0: 'Basicos - solo cuenta easyMoney',
    1: 'Vinculados - nomina y pension',
    2: 'Digitales - cuenta y tarjeta debito',
    3: 'Ahorradores - deposito a largo plazo',
    4: 'Inversores - valores bursatiles',
    5: 'Inactivos - sin vinculacion',
    6: 'Premium - fondos de inversion'
}

df_clust['cluster_name'] = df_clust['cluster'].map(cluster_names)

print("Distribucion final por segmento:")
for c, name in cluster_names.items():
    n = int((df_clust['cluster'] == c).sum())
    print(f"  [{c}] {name:<48} {n:>7,} clientes ({n/len(df_clust)*100:.1f}%)")

## 12. Visualización de los Segmentos

Tres perspectivas complementarias:
1. **Distribucion de clientes** — tamano de cada segmento
2. **Heatmap de perfil** — caracteristicas medias por cluster
3. **PCA 2D** — representacion espacial orientativa (con limitaciones documentadas)

In [ ]:
colors = ['#636EFA','#EF553B','#00CC96','#AB63FA',
          '#FFA15A','#19D3F3','#FF6692']

fig1 = go.Figure(go.Bar(
    x=[cluster_names[i] for i in range(K_FINAL)],
    y=[int((df_clust['cluster'] == i).sum()) for i in range(K_FINAL)],
    marker_color=colors,
    text=[f"{(df_clust['cluster']==i).sum()/len(df_clust)*100:.1f}%"
          for i in range(K_FINAL)],
    textposition='outside'
))
fig1.update_layout(
    title='Distribucion de Clientes por Segmento',
    xaxis_title='Segmento',
    yaxis_title='Numero de Clientes',
    height=500,
    xaxis_tickangle=-20
)
fig1.show()

In [ ]:
heatmap_cols = ['em_acount', 'emc_account', 'payroll_account', 'payroll',
                'credit_card', 'debit_card', 'funds', 'securities',
                'pension_plan', 'long_term_deposit', 'active_customer',
                'total_products']

fig2 = px.imshow(
    profile[heatmap_cols].T,
    labels=dict(x='Segmento', y='Feature', color='Valor medio'),
    x=[cluster_names[i] for i in range(K_FINAL)],
    y=heatmap_cols,
    color_continuous_scale='YlOrRd',
    aspect='auto',
    text_auto='.2f'
)
fig2.update_layout(
    title='Perfil de Segmentos — Media de Features por Cluster',
    height=500,
    xaxis_tickangle=-20
)
fig2.show()

### PCA 2D — Representacion Espacial Orientativa

> **Limitacion importante:** el PCA 2D captura aproximadamente el **36% de la varianza total**
> de las 16 dimensiones. La visualizacion es util para detectar solapamientos evidentes entre
> clusters, pero la separacion o proximidad visual **no refleja la distancia real** en el
> espacio original de 16 features. Clusters bien separados en alta dimension pueden aparecer
> solapados en 2D, y viceversa.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_final_scaled)

var1      = pca.explained_variance_ratio_[0]
var2      = pca.explained_variance_ratio_[1]
total_var = var1 + var2
print(f"Varianza explicada: PC1={var1:.2%}, PC2={var2:.2%}, Total={total_var:.2%}")
print(f"Varianza NO representada en 2D: {1-total_var:.2%} — interpretar visualizacion con cautela")

np.random.seed(RANDOM_STATE)
sample_idx_pca = np.random.choice(len(X_pca), size=50000, replace=False)

df_pca = pd.DataFrame({
    'PC1': X_pca[sample_idx_pca, 0],
    'PC2': X_pca[sample_idx_pca, 1],
    'cluster_name': df_clust['cluster_name'].iloc[sample_idx_pca].values
})

fig3 = px.scatter(
    df_pca, x='PC1', y='PC2',
    color='cluster_name',
    color_discrete_sequence=colors,
    opacity=0.4,
    title=f'Segmentacion — PCA 2D (muestra 50k clientes | varianza explicada: {total_var:.1%})',
    labels={'cluster_name': 'Segmento'},
    hover_data=['cluster_name']
)
fig3.update_traces(marker=dict(size=3))
fig3.update_layout(height=600)
fig3.show()

## 13. Exportación de Resultados para Power BI

In [ ]:
output_dir = '../../data/processed/'
os.makedirs(output_dir, exist_ok=True)

# CSV 1 — segmento por cliente, enriquecido con variables demograficas
customer_segments = df_clust[['pk_cid', 'cluster', 'cluster_name']].copy()
extra_cols = ['pk_cid', 'age_group', 'salary_group', 'segment',
              'gender', 'region_code', 'country_id']
customer_segments = customer_segments.merge(
    df_latest[extra_cols], on='pk_cid', how='left'
)
customer_segments.to_csv(f'{output_dir}customer_segments.csv', index=False)

# CSV 2 — perfil de clusters con accion recomendada
profile_export = profile.copy()
profile_export.index.name = 'cluster_id'
profile_export['cluster_name'] = [cluster_names[i] for i in range(K_FINAL)]

acciones = {
    0: 'Upsell: tarjeta debito y payroll_account',
    1: 'Retencion: evitar fuga, ofrecer fondos de inversion',
    2: 'Cross-sell: domiciliacion nomina y plan de pensiones',
    3: 'Fidelizacion: productos de ahorro complementarios',
    4: 'Premium: fondos de inversion, atencion personalizada',
    5: 'Reactivacion: campana especifica o cierre de cuenta',
    6: 'Retencion VIP: productos exclusivos, gestor personal'
}
profile_export['accion_recomendada'] = [acciones[i] for i in range(K_FINAL)]
profile_export.to_csv(f'{output_dir}cluster_profiles.csv')

print(f"customer_segments.csv  — {len(customer_segments):,} filas")
print(f"  Columnas: {customer_segments.columns.tolist()}")
print()
print(f"cluster_profiles.csv   — {K_FINAL} clusters")
print(f"  Columnas: {profile_export.columns.tolist()}")
print()
print(f"Archivos guardados en: {output_dir}")
print()
print("Muestra customer_segments:")
print(customer_segments.head(5).to_string())

## 14. Conclusiones

### Resumen de la Segmentación

Se han identificado **7 segmentos** con perfiles claramente diferenciados:

| Cluster | Nombre | Clientes | % | Perfil clave | Accion recomendada |
|---------|--------|----------|---|-------------|-------------------|
| 0 | Basicos | 161,026 | 36.3% | Solo cuenta easyMoney, sin tarjeta ni nomina, sin vinculacion adicional (~26 anos) | **Upsell:** tarjeta debito y payroll_account |
| 1 | Vinculados | 16,924 | 3.8% | Nomina + pension + tarjeta debito, alta vinculacion (~4 productos) | **Retencion:** evitar fuga, ofrecer fondos |
| 2 | Digitales | 142,079 | 32.1% | Cuenta + tarjeta debito, activos digitalmente (~1.3 productos, ~33 anos) | **Cross-sell:** domiciliacion nomina y pension_plan |
| 3 | Ahorradores | 5,443 | 1.2% | Deposito a largo plazo, mayor edad (~53 anos), salary medio-alto | **Fidelizacion:** productos de ahorro complementarios |
| 4 | Inversores | 1,637 | 0.4% | Valores bursatiles, salary medio-alto, ~3.2 productos por cliente | **Premium:** fondos de inversion, atencion personalizada |
| 5 | Inactivos | 114,571 | 25.9% | Sin cuenta ni productos, engagement practicamente nulo | **Reactivacion:** campana especifica o cierre |
| 6 | Premium | 1,315 | 0.3% | Fondos de inversion, salary mas alto (~132k EUR), ~48 anos, multiproducto | **Retencion VIP:** productos exclusivos, gestor personal |

### Implicaciones Estrategicas

**Campana Erin — 10,000 emails:**
- **Prioridad 1 → Basicos (36.3%):** objetivo de conversion a Digitales mediante tarjeta debito
- **Prioridad 2 → Digitales (32.1%):** activar domiciliacion de nomina y plan de pensiones
- **Excluir → Inactivos (25.9%):** ROI esperado muy bajo — ausencia total de vinculacion

**Segmentos de alto valor (retencion prioritaria):**
- Vinculados, Inversores y Premium concentran el mayor numero de productos por cliente
- La perdida de estos clientes tiene impacto desproporcionado en el revenue

**Mayor potencial de crecimiento:**
- Basicos + Digitales = **68.4% de la base** con baja vinculacion actual
- Representan la mayor oportunidad de cross-selling a corto plazo

### Calidad del Modelo

| Metrica | Valor | Interpretacion |
|---------|-------|----------------|
| Silhouette (k=7) | ~0.221 | Moderado — habitual en datos bancarios sin clusters muy separados |
| Davies-Bouldin | ~1.289 | Aceptable |
| Cluster minimo | >1,000 clientes | Todos los segmentos tienen masa critica accionable |
| PCA varianza 2D | ~36% | Visualizacion orientativa — no usar para medir separacion real |

> El silhouette moderado no invalida el modelo. En datos de clientes bancarios no existen
> "clusters naturales" perfectamente delimitados. La utilidad del modelo se mide por su
> **interpretabilidad comercial y accionabilidad**, no solo por metricas matematicas.

### Outputs Generados
- `customer_segments.csv` — 442,995 clientes con cluster asignado (para Power BI)
- `cluster_profiles.csv` — perfil medio de los 7 clusters con accion recomendada (para Power BI)